## Problem and Motivation
### Problem Description
This report addresses the regression problem of predicting a song's popularity score on Spotify based on its features, using a dataset of charting songs spanning the 1950s to the 2010s. The popularity score ( `pop` ) is a metric that reflects the popularity of a song on Spotify, ranging from 0 to 100. It is based on the total stream counts weighted towards recent listening activity.
### Motivation
Understanding what makes a song popular has clear commercial value for artists, record labels, and streaming platforms alike. By identifying which measurable charactacteristics of a song relate to its popularity, we can build a model that quantifies these relationships empirically, rather than relying solely on intuition and hopeful speculation.
### Primary Question
**Can we predict a song's Spotify popularity from its audio features and the decade it was released?**
### Data
The dataset is a unification of seven .csv files, each containing songs from a decade from 1950s to 2010s, sourced from Spotify's API via Kaggle. Each row represents a charting song and includes audio features such as energy, danceability, acousticness, and tempo, alongside a popularity score. Files were combined into a single dataset for analysis, with a `decade` column dervied from the source file to perserve chronological information.

The dataset is available publicly on [Kaggle](https://www.kaggle.com/datasets/cnic92/spotify-past-decades-songs-50s10s).
### Limitations
It is important to acknowledge that Spotify's popularity score reflects **_recent_** stream counts rather than a timeless measure of quality. According to Spotify's API documentation, as cited by Musicstax, the popularity score is _"...based, the most part, on the total number of plays the track has had and how recent those plays are."_ ([Musicstax,2024](https://musicat.musicstax.com/spotify-popularity-index))

## Data Loading and Merging
The seven files are combined into a single dataframe with a `decade` column derived from each filename before merging, keeping timeline information that would otherwise be lost.

In [1]:
# Initial import for data ingestion
import pandas as pd

In [2]:
# Read in .csv files
df_1950s = pd.read_csv("data/1950.csv")
df_1960s = pd.read_csv("data/1960.csv")
df_1970s = pd.read_csv("data/1970.csv")
df_1980s = pd.read_csv("data/1980.csv")
df_1990s = pd.read_csv("data/1990.csv")
df_2000s = pd.read_csv("data/2000.csv")
df_2010s = pd.read_csv("data/2010.csv")

# Create list of dfs and decades
dfs = [df_1950s, df_1960s, df_1970s, df_1980s, df_1990s, df_2000s, df_2010s]
decades = [1950, 1960, 1970, 1980, 1990, 2000, 2010]

# Loop through dfs and add decade column
for df in dfs:
    df["decade"] = decades.pop(0)

# Combine the dataset
combined_df = pd.concat(dfs, ignore_index=True)
print(combined_df.shape)

(667, 16)


## Data Cleaning
### Duplicates and Redundant Columns
Seven songs appeared across multiple decade files due to re-releases. Duplicates were identified on `title` and `artist`, retaining the earliest release year to preserve the original recording context. The `Number` column was removed as it was redundant.

In [3]:
# Number column removed — relic from combining datasets
combined_df = combined_df.drop(columns=["Number"])

# 7 duplicate songs identified across decade files — keep earliest release year
combined_df = combined_df.sort_values("year", ascending=True)
combined_df = combined_df.drop_duplicates(subset=["title", "artist"], keep="first")
combined_df = combined_df.reset_index(drop=True)

# Verify no duplicates remain
assert combined_df.duplicated(["title", "artist"]).sum() == 0, "Duplicates remain!"
print(f"Duplicates removed. Dataset: {combined_df.shape[0]} rows × {combined_df.shape[1]} columns.")

Duplicates removed. Dataset: 660 rows × 15 columns.


### Missing Values
A null check revealed missing values only in `top genre`. Rather than dropping these rows, missing genres were labelled `"unknown"`, a valid category a model can learn from. No numeric features contained missing values, so no
imputation was required.

In [4]:
# Missing values found only in `top genre` — labelled "unknown".
combined_df["top genre"] = combined_df["top genre"].fillna("unknown")

assert combined_df.isnull().sum().sum() == 0, "Missing values remain!"
print("No missing values remain.")

No missing values remain.


### Incoherent Years
The assignment notes that some `year` values are incoherent due to re-releases. The `decade` column derived from the source filename is used as the authoritative record of when a song was released. Incoherent years are flagged rather than
corrected, with `decade` used in modelling instead of `year`.

In [6]:
# Keep all columns - add incoherent_year flag column
combined_df["incoherent_year"] = combined_df.apply(
    # If year is less than decade or greater than decade + 9, flag as incoherent
    lambda row: 1 if row["year"] < row["decade"] or row["year"] > row["decade"] + 9 else 0,
    axis=1)   # Apply to each row

print(f"{combined_df['incoherent_year'].sum()} incoherent years ({(combined_df['incoherent_year'].mean()*100):.1f}% of dataset)")

189 incoherent years (28.6% of dataset)


# Exploratory Data Analysis

# Feature Selection and Train/Test Split

# Conclusion and Recommendation